# 01 — Stage, audit, preview and convert TempleRAIL
No dataset downloads, GPU setup, weights, training or Ultralytics. A standard
Colab CPU session with its bundled Pillow is sufficient. Read one archive from
Drive, verify MD5, extract to fast temporary disk and cache completed audits.
Original Drive archive and extracted data are never modified or deleted.
Staging/cache writes are automatic; dataset conversion still requires explicit
confirmation in cell 11. No input annotations.json is needed.


## 1. Repository and Drive
Upload and extract this repository's code to `/content/aquafina-yolo-detector`.
Include src, configs, scripts, requirements and pyproject.toml; no raw data or
weights in the repository ZIP. Mount Drive below. Do not use Run all to approve
conversion: review audit and previews first. Cell numbers include markdown.


In [ ]:
import sys
sys.dont_write_bytecode = True
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/aquafina-yolo-detector')
assert (REPO / 'pyproject.toml').is_file(), 'Extract repository code first'
sys.path.insert(0, str(REPO / 'src'))
ARCHIVE = Path('/content/drive/MyDrive/aquafina-yolo/raw/temple/detection_dataset.tar.gz')
EXPECTED_MD5 = 'fca7260d4785af1dec18aa320fa9fc4a'
STAGE_DIR = Path('/content/temple_stage')
TEMPLE_ROOT = Path('/content/temple_stage/detection_dataset')
CACHE_DIR = Path('/content/drive/MyDrive/aquafina-yolo/cache/temple_audit')
PROCESSED_ROOT = Path('/content/drive/MyDrive/aquafina-yolo/processed/temple')
from aquafina_detector.temple import preview_temple, convert_temple
from aquafina_detector.temple_stage import stage_archive, audit_staged
STAGE = None
AUDIT = None
PREVIEW_SHOWN = False
CONVERSION_RESULT = None
print('Original archive (read-only):', ARCHIVE)
print('Temporary audit input:', TEMPLE_ROOT)
print('Proposed output (not created):', PROCESSED_ROOT)


## 2. Verify archive and stage on temporary disk
Copy the single existing archive from Drive while calculating MD5 and SHA256.
A mismatch stops before extraction. Extract locally, never by copying thousands
of individual source files from Drive. Safe extraction rejects path traversal,
links and special files. An existing complete stage is checked locally before
reuse; interrupted/conflicting stages require a fresh runtime or stage path.
No original archive or extracted Drive dataset is deleted or modified.


In [ ]:
AUDIT = None
PREVIEW_SHOWN = False
CONVERSION_RESULT = None
STAGE = None
STAGE = stage_archive(ARCHIVE, EXPECTED_MD5, STAGE_DIR,
                      progress=lambda message: print(message, flush=True))
assert STAGE.root == TEMPLE_ROOT
print('Verified local input:', STAGE.root)


## 3. Audit locally or reuse the completed verified cache
Decode every image; check image/XML/TXT pairing, dimensions, boxes and classes.
All per-file checks now use temporary disk. Progress prints after every 250
images and on completion. Set REUSE_AUDIT_CACHE=False to force a fresh audit.
A completed report is cached in Drive under the archive MD5; reuse requires
matching archive hashes, audit-code signature, policy and staged-file hashes.
Incomplete, corrupt, stale or blocked results never bypass a fresh audit.
Ignore train.txt for membership: preserve unique val.txt IDs and use
all JPEGImages IDs minus validation. Expect 4,000 train / 870 val / zero overlap.
Known dog and corrupt-train-list warnings do not alone block conversion.
Unexplained class/geometry mismatches and cross-split identical images do block it.


In [ ]:
import json
AUDIT = None
PREVIEW_SHOWN = False
CONVERSION_RESULT = None
assert STAGE is not None, 'Run staging cell 5 first'
REUSE_AUDIT_CACHE = True
AUDIT = audit_staged(STAGE, CACHE_DIR, reuse=REUSE_AUDIT_CACHE,
                     progress=lambda message: print(message, flush=True))
print(json.dumps(AUDIT.audit, indent=2))
print(json.dumps(AUDIT.class_counts, indent=2))
print(json.dumps(AUDIT.anomalies, indent=2))
print('Completed audit cache:', CACHE_DIR / (STAGE.archive_md5 + '.json'))


## 4. Read-only preview from temporary disk
Green: Aquafina boxes kept. Orange: competitor boxes become background.
Red: XML dog box excluded. Samples prioritize the dog image and available
positive/mixed/negative subsets. Inspect all reported anomalies, not just these
samples. Source annotations do not prove visible brand identity in every image.


In [ ]:
from IPython.display import display
PREVIEW_SHOWN = False
assert AUDIT is not None, 'Run audit cell 7 first'
previews = preview_temple(AUDIT, limit=8)
for caption, image in previews:
    print(caption)
    display(image)
PREVIEW_SHOWN = bool(previews)
print('Preview displayed; nothing saved. Blocking errors:', AUDIT.audit['blocking_errors'])


## 5. Explicit confirmation before converted-dataset writes
After reviewing audit and preview, change CONFIRM_CONVERSION to True and run
cell 11. Keep it False to skip conversion. A clean audit and completed preview
are required. Conversion writes exclusively under PROCESSED_ROOT, copies images
(no raw symlinks), and checks raw hashes before/after. Existing output is never
overwritten; use a new child/version path for another conversion.

Aquafina Darknet class 0 becomes model class 0 / COCO category 1. Keep every
image; competitor-only images have empty annotations, and mixed images retain
Aquafina boxes only. Dog objects are excluded. No test split is manufactured.


In [ ]:
CONFIRM_CONVERSION = False
if not CONFIRM_CONVERSION:
    print('Conversion not confirmed; no converted dataset written. Staging/audit cache are retained.')
else:
    assert AUDIT is not None and PREVIEW_SHOWN, 'Run audit cell 7 and preview cell 9 first'
    assert AUDIT.audit['blocking_errors'] == 0, 'Resolve blocking audit errors before conversion'
    CONVERSION_RESULT = convert_temple(AUDIT, PROCESSED_ROOT, confirm=True)
    print(json.dumps(CONVERSION_RESULT, indent=2))


## 6. Read back completed outputs
audit.json, anomaly_report.json, class_counts.json, train_manifest.json,
val_manifest.json, split_manifest.json and conversion.json accompany the
annotations/train.json, annotations/val.json and train2017/val2017 images.
The conversion marker is written last; its absence means an incomplete output.
Stop here. Training is a separate future action, not part of this notebook.


In [ ]:
if CONVERSION_RESULT is None:
    print('No conversion performed in this session.')
else:
    from aquafina_detector.common import read_json
    marker = read_json(PROCESSED_ROOT / 'conversion.json')
    print({key: marker[key] for key in ('status', 'train_images', 'val_images', 'raw_sha256_before_after_equal')})
    print('Files:', sorted(path.name for path in PROCESSED_ROOT.iterdir()))
